In [ ]:
# Adicionem aqui as bibliotecas necessárias
import pandas as pd
import os
from PIL import Image
from torchvision import datasets, transforms, models
import cv2
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path = '/content/drive/MyDrive/Projeto LIPAI/Displastia'
dataset_name = "Displastia"

In [ ]:
import json
import os

def save_metrics_json(base_dir,
                     arch_name, mode, aug,
                     seed, acc, f1_macro, f1_weighted):

    dir_path = f'{base_dir}/{dataset_name}/results'
    os.makedirs(dir_path, exist_ok=True)

    results = {
        "architecture": arch_name,
        "mode": mode,
        "aug": aug,
        "seed": seed,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

    aug_str = "non_aug" if not aug else "aug"

    filename = f"{arch_name}_{mode}_{aug_str}_seed{seed}.json"
    path = os.path.join(dir_path, filename)

    with open(path, 'w') as f:
        json.dump(results, f, indent=4)

    print(f"Métricas salvas em: {path}")

# 1. Carregamento dos Dados

In [ ]:
manifest_d = pd.read_csv(f'{path}/manifest.csv')
images_d = datasets.ImageFolder(root = f'{path}/dataset')

Divisão dos sets de acordo com o manifest

In [ ]:
base_path = f'{path}/dataset'

def load_images_from_df(dataframe):
    images = []
    labels = []

    for idx, row in dataframe.iterrows():
        # Constrói o caminho completo: dataset/healthy/imagem.tif
        img_path = os.path.join(base_path, row['path'])

        # Leitura da imagem
        img = cv2.imread(img_path)
        if img is not None:
            images.append(img)
            labels.append(row['label_number'])
        else:
            print(f"Erro ao carregar: {img_path}")

    return np.array(images), np.array(labels)

# 3. Criar as divisões diretamente do CSV
train_df = manifest_d[manifest_d['sets'] == 'train']
val_df   = manifest_d[manifest_d['sets'] == 'val']
test_df  = manifest_d[manifest_d['sets'] == 'test']

# 4. Carregar os dados para as variáveis
print("Carregando imagens de treino...")
x_train, y_train = load_images_from_df(train_df)

print("Carregando imagens de validação...")
x_val, y_val = load_images_from_df(val_df)

print("Carregando imagens de teste...")
x_test, y_test = load_images_from_df(test_df)

# Conferindo os formatos
print("\nDivisão concluída:")
print(f"Treino: {x_train.shape}, {y_train.shape}")
print(f"Val:    {x_val.shape}, {y_val.shape}")
print(f"Teste:  {x_test.shape}, {y_test.shape}")

## Transformações e Data Augmentation

Não queremos distorções brucas nas imagens histológicas, portanto, escolhemos aplicar apenas 3 operações geometricas.

* RandomHorizontalFlip: Espelhamento horizontal.
* RandomVerticalFlip: Espelhamento vertical.
* RandomRotation: Rotação suave de 15 graus.

In [ ]:
transform_basic = transforms.Compose([ # Básica, sem augmentation
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

transform_augmentation = transforms.Compose([ # Com augmentation
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# 2. Aplicação dos Algoritmos

## a) ConvNeXt V2 Atto


### FS - From Scratch

#### Augmentation

##### Seed 42

Avaliação

In [ ]:
save_metrics_json(path,
                  dataset_name,
                  arch_name = "ConvNeXt V2 Atto",
                  mode = "FS",
                  aug = True,
                  seed = "42",
                  acc = ..., # acurácia da execução
                  f1_macro = ..., # F1-score macro da execução
                  f1_weighted = ... # F1-score weighted da execução
                  )

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 123

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "FS", True, 123, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 2025

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "FS", True, 2025, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

In [ ]:
# MÉDIA E DESVIO PADRÃO DAS 3 REPETIÇÕES

# Métricas obrigatórias para média e desvio padrão:
# ● Acurácia.
# ● F1-score macro.
# ● F1-score weighted.

#### Sem Augmentation

##### Seed 42

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "FS", False, 42, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 123

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "FS", False, 123, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 2025

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "FS", False, 2025, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

In [ ]:
# MÉDIA E DESVIO PADRÃO DAS 3 REPETIÇÕES

# Métricas obrigatórias para média e desvio padrão:
# ● Acurácia.
# ● F1-score macro.
# ● F1-score weighted.

### PT-FC – Pré-treinado com Backbone Congelado

#### Augmentation

##### Seed 42

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-FC", True, 42, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 123

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-FC", True, 123, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 2025

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-FC", True, 2025, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

In [ ]:
# MÉDIA E DESVIO PADRÃO DAS 3 REPETIÇÕES

# Métricas obrigatórias para média e desvio padrão:
# ● Acurácia.
# ● F1-score macro.
# ● F1-score weighted.

#### Sem Augmentation

##### Seed 42

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-FC", False, 42, acc, f1_macro, f1_weighted)


In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 123

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-FC", False, 123, acc, f1_macro, f1_weighted)


In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 2025

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-FC", False, 2025, acc, f1_macro, f1_weighted)


In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

In [ ]:
# MÉDIA E DESVIO PADRÃO DAS 3 REPETIÇÕES

# Métricas obrigatórias para média e desvio padrão:
# ● Acurácia.
# ● F1-score macro.
# ● F1-score weighted.

### PT-ALL – Pré-treinado com Fine-tuning Completo

#### Augmentation

##### Seed 42

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-ALL", True, 42, acc, f1_macro, f1_weighted)


In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 123

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-ALL", True, 123, acc, f1_macro, f1_weighted)


In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 2025

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-ALL", True, 2025, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

In [ ]:
# MÉDIA E DESVIO PADRÃO DAS 3 REPETIÇÕES

# Métricas obrigatórias para média e desvio padrão:
# ● Acurácia.
# ● F1-score macro.
# ● F1-score weighted.

#### Sem Augmentation

##### Seed 42

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-ALL", False, 42, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 123

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-ALL", False, 123, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 2025

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-ALL", False, 2025, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

In [ ]:
# MÉDIA E DESVIO PADRÃO DAS 3 REPETIÇÕES

# Métricas obrigatórias para média e desvio padrão:
# ● Acurácia.
# ● F1-score macro.
# ● F1-score weighted.

## b) ConvNeXt V2 Pico


### FS - From Scratch

#### Augmentation

##### Seed 42

Avaliação

In [ ]:
save_metrics_json(path,
                  dataset_name,
                  arch_name = "ConvNeXt V2 Atto",
                  mode = "FS",
                  aug = True,
                  seed = "42",
                  acc = ..., # acurácia da execução
                  f1_macro = ..., # F1-score macro da execução
                  f1_weighted = ... # F1-score weighted da execução
                  )

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 123

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "FS", True, 123, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 2025

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "FS", True, 2025, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

In [ ]:
# MÉDIA E DESVIO PADRÃO DAS 3 REPETIÇÕES

# Métricas obrigatórias para média e desvio padrão:
# ● Acurácia.
# ● F1-score macro.
# ● F1-score weighted.

#### Sem Augmentation

##### Seed 42

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "FS", False, 42, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 123

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "FS", False, 123, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 2025

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "FS", False, 2025, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

In [ ]:
# MÉDIA E DESVIO PADRÃO DAS 3 REPETIÇÕES

# Métricas obrigatórias para média e desvio padrão:
# ● Acurácia.
# ● F1-score macro.
# ● F1-score weighted.

### PT-FC – Pré-treinado com Backbone Congelado

#### Augmentation

##### Seed 42

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-FC", True, 42, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 123

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-FC", True, 123, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 2025

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-FC", True, 2025, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

In [ ]:
# MÉDIA E DESVIO PADRÃO DAS 3 REPETIÇÕES

# Métricas obrigatórias para média e desvio padrão:
# ● Acurácia.
# ● F1-score macro.
# ● F1-score weighted.

#### Sem Augmentation

##### Seed 42

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-FC", False, 42, acc, f1_macro, f1_weighted)


In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 123

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-FC", False, 123, acc, f1_macro, f1_weighted)


In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 2025

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-FC", False, 2025, acc, f1_macro, f1_weighted)


In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

In [ ]:
# MÉDIA E DESVIO PADRÃO DAS 3 REPETIÇÕES

# Métricas obrigatórias para média e desvio padrão:
# ● Acurácia.
# ● F1-score macro.
# ● F1-score weighted.

### PT-ALL – Pré-treinado com Fine-tuning Completo

#### Augmentation

##### Seed 42

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-ALL", True, 42, acc, f1_macro, f1_weighted)


In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 123

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-ALL", True, 123, acc, f1_macro, f1_weighted)


In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 2025

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-ALL", True, 2025, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

In [ ]:
# MÉDIA E DESVIO PADRÃO DAS 3 REPETIÇÕES

# Métricas obrigatórias para média e desvio padrão:
# ● Acurácia.
# ● F1-score macro.
# ● F1-score weighted.

#### Sem Augmentation

##### Seed 42

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-ALL", False, 42, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 123

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-ALL", False, 123, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

##### Seed 2025

In [ ]:
save_metrics_json(path, dataset_name, "ConvNeXt V2 Atto", "PT-ALL", False, 2025, acc, f1_macro, f1_weighted)

In [ ]:
# aqui devem vir as curvas de aprendizado:
# - loss
# - acurácia

In [ ]:
# MÉDIA E DESVIO PADRÃO DAS 3 REPETIÇÕES

# Métricas obrigatórias para média e desvio padrão:
# ● Acurácia.
# ● F1-score macro.
# ● F1-score weighted.

# 3. Comparativos Globais



In [ ]:
# Para cada dataset, o grupo deverá apresentar comparações globais usando: <br>
#  * Gráfico de barras do F1-score por arquitetura, modo de treinamento e uso de augmentation. <br>
#  * Tabela com média e desvio padrão das repetições. <br>
#  * Gráfico com número de parâmetros por arquitetura. <br>
#  * Gráfico com complexidade computacional em GFLOPs considerando entrada 224 × 224.